# MathScy MoE Training Pipeline

## Architecture: Domain-Specialized LoRA-MoE with Shared Expert

This notebook implements a **DeepSeekMoE-inspired** architecture where:
- A **shared expert** learns general mathematical reasoning
- **Domain-specific LoRA experts** specialize in different math domains
- A **learned router** assigns math content to the appropriate expert
- **Auxiliary-loss-free balancing** (ICLR 2025) ensures stable training

### Key Design Decisions:
1. **Base model**: DeepSeek-Math-7B (state-of-the-art math reasoning)
2. **MoE approach**: LoRA adapters as experts (memory efficient for 72hr GPU budget)
3. **Router**: Sigmoid gating with bias-term load balancing
4. **Expert domains**: Aligned with ArXiv math categories
5. **Training objective**: Next-token prediction on domain-specific math content

### GPU Requirements:
- DGX A100 (8x80GB): Full training with DeepSpeed ZeRO-3
- DGX H100 (8x80GB): Faster training, can use larger batch sizes

### Resumability:
- Checkpoints saved every 500 steps
- Training can resume from any checkpoint
- All hyperparameters saved in config

In [ ]:
import os
import sys
import json
import math
import time
import random
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# HuggingFace
os.environ['HF_HOME'] = '/scratch/ctoxtli/cache'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, 
    TrainingArguments, Trainer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    PeftModel, TaskType,
)

# Config
with open('/scratch/ctoxtli/moexp/configs/project_config.json') as f:
    config = json.load(f)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'Number of GPUs: {torch.cuda.device_count()}')

## 1. Configuration

In [ ]:
@dataclass
class MoEConfig:
    """Configuration for MathScy MoE training."""
    # Base model
    base_model_name: str = 'deepseek-ai/deepseek-math-7b-instruct'
    
    # MoE architecture
    num_domain_experts: int = 8
    expert_domains: List[str] = field(default_factory=lambda: [
        'algebraic_geometry', 'combinatorics', 'number_theory',
        'analysis', 'algebra', 'geometry_topology',
        'probability_statistics', 'computational_math'
    ])
    use_shared_expert: bool = True  # DeepSeekMoE-style shared expert
    top_k_experts: int = 2  # Number of experts activated per token
    router_type: str = 'sigmoid'  # 'sigmoid' or 'softmax'
    
    # LoRA config
    lora_rank: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ])
    
    # Training
    learning_rate: float = 2e-4
    batch_size_per_device: int = 4
    gradient_accumulation_steps: int = 8
    num_epochs: int = 3
    max_seq_length: int = 4096
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    
    # Checkpointing
    checkpoint_dir: str = '/scratch/ctoxtli/moexp/models/moe_checkpoints'
    save_steps: int = 500
    logging_steps: int = 10
    
    # Router training
    router_lr: float = 1e-3
    balance_loss_weight: float = 0.0  # Auxiliary-loss-free approach
    router_bias_update_speed: float = 0.001  # For bias-term balancing
    
    # Quantization
    use_4bit: bool = True
    bnb_4bit_compute_dtype: str = 'bfloat16'
    bnb_4bit_quant_type: str = 'nf4'

moe_config = MoEConfig()
print(f'Config: {json.dumps({k: str(v) for k, v in moe_config.__dict__.items()}, indent=2)}')

## 2. Math Domain Router

The router learns to assign math content to the appropriate domain expert.
Uses sigmoid gating (independent per-expert) with bias-term load balancing.

In [ ]:
class MathDomainRouter(nn.Module):
    """
    Sigmoid-based router with auxiliary-loss-free balancing.
    
    Each expert has an independent sigmoid gate.
    Load balancing is achieved by adjusting bias terms based on
    running expert utilization statistics (no aux loss needed).
    
    Reference: ICLR 2025 - Auxiliary-Loss-Free Load Balancing
    """
    def __init__(self, hidden_size: int, num_experts: int, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # Routing projection
        self.gate = nn.Linear(hidden_size, num_experts, bias=True)
        
        # Learnable bias for load balancing (updated based on utilization)
        self.register_buffer('expert_load_ema', torch.zeros(num_experts))
        self.register_buffer('balance_bias', torch.zeros(num_experts))
        self.balance_update_speed = 0.001
        
    def forward(self, hidden_states: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            hidden_states: [batch, seq_len, hidden_size]
        Returns:
            router_weights: [batch, seq_len, num_experts] - soft routing weights
            selected_experts: [batch, seq_len, top_k] - indices of selected experts
        """
        # Compute routing logits
        logits = self.gate(hidden_states)  # [B, T, E]
        
        # Add balance bias (detached from gradient to prevent it from affecting main loss)
        balanced_logits = logits + self.balance_bias.detach()
        
        # Sigmoid gating (independent per expert)
        scores = torch.sigmoid(balanced_logits)  # [B, T, E]
        
        # Select top-k experts
        topk_scores, topk_indices = torch.topk(scores, k=self.top_k, dim=-1)  # [B, T, k]
        
        # Normalize top-k scores
        topk_weights = topk_scores / (topk_scores.sum(dim=-1, keepdim=True) + 1e-9)
        
        # Update load statistics (only during training)
        if self.training:
            with torch.no_grad():
                # Count how many tokens each expert was selected for
                expert_mask = torch.zeros_like(scores)
                expert_mask.scatter_(-1, topk_indices, 1.0)
                load = expert_mask.sum(dim=[0, 1])  # [E]
                total = load.sum()
                
                # EMA update
                load_fraction = load / (total + 1e-9)
                self.expert_load_ema = 0.99 * self.expert_load_ema + 0.01 * load_fraction
                
                # Update balance bias to encourage underutilized experts
                target_load = 1.0 / self.num_experts
                self.balance_bias += self.balance_update_speed * (target_load - self.expert_load_ema)
        
        return topk_weights, topk_indices

# Test router
router = MathDomainRouter(hidden_size=256, num_experts=8, top_k=2)
test_input = torch.randn(2, 10, 256)
weights, indices = router(test_input)
print(f'Router output shapes: weights={weights.shape}, indices={indices.shape}')
print(f'Expert utilization: {router.expert_load_ema}')

## 3. Domain-Specialized LoRA MoE Model

The model uses multiple LoRA adapters as domain experts, with a shared backbone.

In [ ]:
class MathScy_MoE_Model(nn.Module):
    """
    MathScy Mixture-of-LoRA-Experts model.
    
    Architecture:
    - Frozen base LLM (quantized)
    - Shared LoRA expert (always active)
    - N domain-specific LoRA experts (top-k selected per token)
    - Learned sigmoid router with bias-term load balancing
    
    This approach is memory-efficient (only LoRA params are trained)
    and can be trained within 72 hours on a DGX A100/H100.
    """
    
    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config
        
        # Load quantized base model
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=config.use_4bit,
            bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
            bnb_4bit_quant_type=config.bnb_4bit_quant_type,
            bnb_4bit_use_double_quant=True,
        )
        
        print(f'Loading base model: {config.base_model_name}')
        self.base_model = AutoModelForCausalLM.from_pretrained(
            config.base_model_name,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True,
        )
        self.base_model = prepare_model_for_kbit_training(self.base_model)
        
        # Get hidden size from base model
        hidden_size = self.base_model.config.hidden_size
        
        # Shared expert (LoRA adapter)
        if config.use_shared_expert:
            shared_lora_config = LoraConfig(
                r=config.lora_rank,
                lora_alpha=config.lora_alpha,
                target_modules=config.lora_target_modules,
                lora_dropout=config.lora_dropout,
                bias='none',
                task_type=TaskType.CAUSAL_LM,
            )
            self.base_model = get_peft_model(self.base_model, shared_lora_config, adapter_name='shared')
        
        # Domain expert LoRA adapters
        for i, domain in enumerate(config.expert_domains):
            expert_lora_config = LoraConfig(
                r=config.lora_rank // 2,  # Smaller rank for domain experts
                lora_alpha=config.lora_alpha // 2,
                target_modules=config.lora_target_modules,
                lora_dropout=config.lora_dropout,
                bias='none',
                task_type=TaskType.CAUSAL_LM,
            )
            self.base_model.add_adapter(domain, expert_lora_config)
        
        # Router
        self.router = MathDomainRouter(
            hidden_size=hidden_size,
            num_experts=config.num_domain_experts,
            top_k=config.top_k_experts,
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            config.base_model_name,
            trust_remote_code=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f'Model loaded with {config.num_domain_experts} domain experts + shared expert')
        self._print_trainable_params()
    
    def _print_trainable_params(self):
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
    
    def forward_with_routing(self, input_ids, attention_mask=None, labels=None):
        """
        Forward pass with MoE routing.
        
        For efficiency, we use a simplified approach:
        1. Run the shared expert on all tokens
        2. Use router to determine domain expert weights
        3. Blend domain expert outputs with routing weights
        
        In production, this would use proper sparse expert execution.
        For LoRA-based MoE, we can merge/unmerge adapters dynamically.
        """
        # Get hidden states from shared expert
        self.base_model.set_adapter('shared')
        shared_output = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        
        # Use last hidden state for routing
        hidden_states = shared_output.hidden_states[-1]  # [B, T, H]
        routing_weights, expert_indices = self.router(hidden_states)
        
        # For training: compute weighted loss across selected experts
        # This is the simplified version - in practice, would use sparse execution
        total_loss = torch.tensor(0.0, device=input_ids.device)
        
        if labels is not None:
            # Shared expert loss (always included)
            shift_logits = shared_output.logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            shared_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
            total_loss = 0.5 * shared_loss  # Shared expert weight
            
            # Domain expert losses (top-k weighted)
            for k in range(self.config.top_k_experts):
                # Get the k-th selected expert for each token
                expert_idx = expert_indices[:, :, k]  # [B, T]
                expert_weight = routing_weights[:, :, k]  # [B, T]
                
                # For simplicity, use the most common expert in the batch
                most_common_expert = expert_idx.mode(dim=1).values.mode().values.item()
                expert_name = self.config.expert_domains[most_common_expert]
                
                self.base_model.set_adapter(expert_name)
                expert_output = self.base_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
                
                expert_logits = expert_output.logits[..., :-1, :].contiguous()
                expert_loss = F.cross_entropy(
                    expert_logits.view(-1, expert_logits.size(-1)),
                    shift_labels.view(-1),
                    ignore_index=-100,
                )
                
                # Weight by average routing weight for this expert
                avg_weight = expert_weight.mean()
                total_loss += avg_weight * expert_loss * 0.5
        
        return {
            'loss': total_loss,
            'logits': shared_output.logits,
            'routing_weights': routing_weights,
            'expert_indices': expert_indices,
        }

print('MathScy_MoE_Model class defined.')
print('Note: Model loading requires GPU. On CPU, we can test the router and data pipeline only.')

## 4. Dataset for MoE Training

In [ ]:
class MathMoEDataset(Dataset):
    """
    Dataset for MathScy MoE training.
    Each entry includes:
    - Mathematical text (theorem/lemma/definition)
    - Domain label (for router supervision)
    - Task type (conjecture/proof/counterexample)
    """
    
    def __init__(self, data_path: str, tokenizer, max_length: int = 4096):
        self.data = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Domain to expert index mapping
        self.domain_to_idx = {
            d: i for i, d in enumerate(MoEConfig().expert_domains)
        }
        
        with open(data_path) as f:
            for line in f:
                entry = json.loads(line)
                self.data.append(entry)
        
        print(f'Loaded {len(self.data)} training examples')
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        entry = self.data[idx]
        
        # Format as instruction-response
        text = f"""### Instruction:\n{entry.get('instruction', '')}\n\n### Response:\n{entry.get('response', '')}"""
        
        # Tokenize
        encoded = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)
        
        # Labels: mask padding tokens
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        
        # Domain expert index
        domain = entry.get('domain', 'other')
        domain_idx = self.domain_to_idx.get(domain, 0)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'domain_idx': domain_idx,
        }

print('MathMoEDataset class defined.')

## 5. Training Loop with Checkpointing

In [ ]:
def train_moe_model(config: MoEConfig, data_path: str, resume_from: str = None):
    """
    Main training function with full checkpointing and resumability.
    """
    # Initialize model
    model = MathScy_MoE_Model(config)
    tokenizer = model.tokenizer
    
    # Dataset
    dataset = MathMoEDataset(data_path, tokenizer, config.max_seq_length)
    
    # Split train/val
    val_size = min(100, len(dataset) // 10)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    
    print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=config.checkpoint_dir,
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size_per_device,
        per_device_eval_batch_size=config.batch_size_per_device,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        warmup_ratio=config.warmup_ratio,
        logging_steps=config.logging_steps,
        save_steps=config.save_steps,
        save_total_limit=5,
        evaluation_strategy='steps',
        eval_steps=config.save_steps,
        load_best_model_at_end=True,
        fp16=True if device.type == 'cuda' else False,
        bf16=False,  # Set True for H100
        dataloader_num_workers=4,
        report_to='none',  # Set to 'wandb' when available
        resume_from_checkpoint=resume_from,
        gradient_checkpointing=True,
    )
    
    # Trainer
    trainer = Trainer(
        model=model.base_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )
    
    # Train
    print('Starting training...')
    train_result = trainer.train(resume_from_checkpoint=resume_from)
    
    # Save final model
    final_path = os.path.join(config.checkpoint_dir, 'final')
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)
    
    # Save router
    torch.save(model.router.state_dict(), os.path.join(final_path, 'router.pt'))
    
    print(f'Training complete. Model saved to {final_path}')
    return model, trainer

print('Training function defined.')
print('\nTo start training on GPU:')
print('  model, trainer = train_moe_model(moe_config, data_path)')

## 6. Alternative Approach: Branch-Train-Mix

If individual LoRA adapters don't specialize well enough,
we can use Branch-Train-Mix (Sukhbaatar et al., 2024):
1. Branch: Create copies of the base model
2. Train: Fine-tune each copy on domain-specific data  
3. Mix: Combine trained models into an MoE with learned routing

In [ ]:
def branch_train_mix(
    base_model_name: str,
    domain_data_paths: Dict[str, str],
    output_dir: str,
    config: MoEConfig,
):
    """
    Branch-Train-Mix approach for creating domain experts.
    
    Phase 1 (Branch-Train): Fine-tune separate LoRA adapters per domain
    Phase 2 (Mix): Learn routing weights to combine experts
    """
    
    # Phase 1: Train domain-specific adapters
    for domain, data_path in domain_data_paths.items():
        print(f'\n=== Training {domain} expert ===')
        
        domain_config = LoraConfig(
            r=config.lora_rank,
            lora_alpha=config.lora_alpha,
            target_modules=config.lora_target_modules,
            lora_dropout=config.lora_dropout,
            bias='none',
            task_type=TaskType.CAUSAL_LM,
        )
        
        # Load fresh base model for each domain
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type='nf4',
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True,
        )
        model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, domain_config)
        
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Train
        domain_output = os.path.join(output_dir, f'expert_{domain}')
        training_args = TrainingArguments(
            output_dir=domain_output,
            num_train_epochs=config.num_epochs,
            per_device_train_batch_size=config.batch_size_per_device,
            gradient_accumulation_steps=config.gradient_accumulation_steps,
            learning_rate=config.learning_rate,
            warmup_ratio=config.warmup_ratio,
            save_steps=config.save_steps,
            save_total_limit=2,
            fp16=True,
            gradient_checkpointing=True,
            report_to='none',
        )
        
        dataset = MathMoEDataset(data_path, tokenizer, config.max_seq_length)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset,
        )
        trainer.train()
        model.save_pretrained(os.path.join(domain_output, 'final'))
        
        # Free memory
        del model, trainer
        torch.cuda.empty_cache()
    
    # Phase 2: Train router on combined data
    print('\n=== Phase 2: Training MoE Router ===')
    # Load all expert adapters and train routing
    # This will be implemented after Phase 1 completes
    
    print('Branch-Train-Mix complete.')

print('Branch-Train-Mix function defined.')

## 7. CPU-Compatible Testing (No GPU Required)

In [ ]:
# Test the router and data pipeline on CPU
print('=== CPU Testing ===')

# Test router
router = MathDomainRouter(hidden_size=256, num_experts=8, top_k=2)

# Simulate training data
for step in range(100):
    fake_hidden = torch.randn(4, 32, 256)
    weights, indices = router(fake_hidden)

print(f'After 100 steps:')
print(f'  Expert load EMA: {router.expert_load_ema.numpy().round(4)}')
print(f'  Balance bias: {router.balance_bias.numpy().round(4)}')
print(f'  Router weights shape: {weights.shape}')
print(f'  Expert indices shape: {indices.shape}')

# Verify load balancing works
max_load = router.expert_load_ema.max().item()
min_load = router.expert_load_ema.min().item()
print(f'  Load balance ratio (max/min): {max_load/min_load:.2f} (target: ~1.0)')

print('\n=== CPU tests passed ===')